# Convert the CHP-134 platemap Excel files to layout-level CSVs

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Define paths
platemap_file = Path("orig_xlsx_files/CHP-134_REPO1_PlateMaps_20260217.xlsx").resolve(
    strict=True
)
corrected_mapping_file = Path(
    "orig_xlsx_files/PedMap_CHP-134_PlateMapping_corrected.xlsx"
).resolve(strict=True)

# Set output folder for platemaps
metadata_folder = Path("metadata")
metadata_folder.mkdir(exist_ok=True)  # Ensure metadata folder exists

In [3]:
# Load the source platemap workbook and the corrected plate-layout mapping workbook
platemap_df = pd.read_excel(platemap_file)
corrected_mapping_df = pd.read_excel(corrected_mapping_file)

# Print basic info about the loaded dataframes
print("Platemap DataFrame:")
display(platemap_df.head())
print("\nCorrected Mapping DataFrame:")
display(corrected_mapping_df.head())

Platemap DataFrame:


,Plate Barcode,Well Position,Batch Id,Concentration (mg/mL),Concentration (ug/mL),Concentration (mM),Concentration (uM),Volume (uL),Volume (nL),Solvent,Formula Weight,Molecular Weight
0,BR00149332,A01,NaN,NaN,NaN,NaN,NaN,0.00,0,NaN,NaN,NaN
1,BR00149332,A02,NaN,NaN,NaN,NaN,NaN,0.04,40,DMSO,NaN,NaN
2,BR00149332,A03,BRD-K88308881-001-08-9,1.651891,1651.891,10.0,10000.0,0.04,40,DMSO,165.1891,165.078979
3,BR00149332,A04,BRD-K13797099-001-07-9,3.494050,3494.050,10.0,10000.0,0.04,40,DMSO,349.4050,349.109627
4,BR00149332,A05,BRD-A93963402-001-01-6,4.885699,4885.699,10.0,10000.0,0.04,40,DMSO,488.5699,488.241018



Corrected Mapping DataFrame:


,SourceBarcode,Plate Map Name,DestinationBarcode,REPO Plate
0,LC00010516,S-B-REPO1_2025-006-1,BR00149332,REPO1 - Plate 6
1,LC00010516,S-B-REPO1_2025-006-1,BR00149333,REPO1 - Plate 6
2,LC00010516,S-B-REPO1_2025-006-1,Assay Plate_1_3,REPO1 - Plate 6
3,LC00010521,S-B-REPO1_2025-007-1,Assay Plate_1_4,REPO1 - Plate 7
4,LC00010521,S-B-REPO1_2025-007-1,Assay Plate_1_5,REPO1 - Plate 7


In [4]:
# Basic validation
required_platemap_columns = {"Plate Barcode", "Well Position"}
required_mapping_columns = {"Plate Map Name", "DestinationBarcode"}

# Check for missing columns in both dataframes
missing_platemap_columns = required_platemap_columns - set(platemap_df.columns)
missing_mapping_columns = required_mapping_columns - set(corrected_mapping_df.columns)

# Raise errors if required columns are missing
if missing_platemap_columns:
    raise ValueError(
        f"Missing columns in {platemap_file.name}: {sorted(missing_platemap_columns)}"
    )
if missing_mapping_columns:
    raise ValueError(
        f"Missing columns in {corrected_mapping_file.name}: {sorted(missing_mapping_columns)}"
    )

# Check for duplicate destination barcodes in the corrected mapping dataframe
if corrected_mapping_df["DestinationBarcode"].duplicated().any():
    duplicated_barcodes = corrected_mapping_df.loc[
        corrected_mapping_df["DestinationBarcode"].duplicated(), "DestinationBarcode"
    ].tolist()
    raise ValueError(f"Duplicate destination barcodes found: {duplicated_barcodes}")

# Check for consistency between source plate barcodes and mapped destination barcodes
source_plate_barcodes = set(platemap_df["Plate Barcode"].dropna().unique())
mapped_plate_barcodes = set(
    corrected_mapping_df["DestinationBarcode"].dropna().unique()
)

In [5]:
# Create one platemap CSV per layout (3 destination plates per layout)
barcode_platemap = []

for plate_map_name, layout_mapping_df in corrected_mapping_df.groupby(
    "Plate Map Name", sort=False
):
    destination_barcodes = layout_mapping_df["DestinationBarcode"].dropna().tolist()

    # The assay plates are missing from the platemap workbook, so use the first real plate
    template_plate_barcode = next(
        (
            barcode
            for barcode in destination_barcodes
            if barcode in source_plate_barcodes
        ),
        None,
    )

    if template_plate_barcode is None:
        raise ValueError(f"No source platemap found for layout {plate_map_name}")

    layout_platemap_df = platemap_df[
        platemap_df["Plate Barcode"] == template_plate_barcode
    ].copy()
    layout_platemap_df["Plate Barcode"] = plate_map_name

    output_file = metadata_folder / f"{plate_map_name}_platemap.csv"
    layout_platemap_df.to_csv(output_file, index=False)
    print(
        f"Saved {output_file} using template plate {template_plate_barcode} "
        f"for {len(destination_barcodes)} destination plates."
    )

    for barcode in destination_barcodes:
        barcode_platemap.append(
            {"Plate Barcode": barcode, "File Name": output_file.name}
        )

unused_source_plates = sorted(source_plate_barcodes - mapped_plate_barcodes)
if unused_source_plates:
    print(f"Source plates not used by the corrected mapping: {unused_source_plates}")

Saved metadata/S-B-REPO1_2025-006-1_platemap.csv using template plate BR00149332 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-007-1_platemap.csv using template plate BR00149355 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-008-1_platemap.csv using template plate BR00149356 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-009-01_platemap.csv using template plate BR00149359 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-001-1_platemap.csv using template plate BR00149543 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-002-1_platemap.csv using template plate BR00149546 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-003-1_platemap.csv using template plate BR00149549 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-004-1_platemap.csv using template plate BR00149555 for 3 destination plates.
Saved metadata/S-B-REPO1_2025-005-1_platemap.csv using template plate BR00149558 for 3 destination plates.
Source plates not used by the correc

In [6]:
# Create a DataFrame for barcode and file mapping
barcode_platemap_df = pd.DataFrame(barcode_platemap)
barcode_platemap_file = metadata_folder / "barcode_platemap.csv"
barcode_platemap_df.to_csv(barcode_platemap_file, index=False)
print(f"Saved barcode mapping file: {barcode_platemap_file}")

Saved barcode mapping file: metadata/barcode_platemap.csv
